# Week 4: Make Data Model Ready

## ===== WEEK 4 START =====

**Project:** Airline Passenger Demand Forecasting using BTS T-100 Domestic Segment(All Carriers)

**Week 4 objectives**
- Process Week 3 training, validation, test partitions.
- Treat missing/invalid inputs and skewed numeric variables.
- Engineer past-only history and calendar features.
- Encode high-cardinality categorical predictors.
- Fit all learned preprocessing parameters on **training data only** and apply them unchanged to validation//test data.
- Make reproducibility/audit artifacts for the repository.

## 1. Environment and reusable Week 4 source

- Implementation stored in `Src/week4_src_model_ready.py`. 
- Notebook calls that source directly so the submitted notebook / Python code execute same processing logic.

In [ ]:
from pathlib import Path
from IPython.display import display, Image
import importlib.util
import sys

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "Data").exists() and (path / "Src").exists() and (path / "Notebooks").exists():
            return path
    raise FileNotFoundError("Could not locate the project root.")

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "Src"
WEEK4_REPORTS_DIR = PROJECT_ROOT / "Reports" / "week4"
WEEK4_TABLES_DIR = WEEK4_REPORTS_DIR / "week4_tables"
WEEK4_FIGURES_DIR = WEEK4_REPORTS_DIR / "week4_figures"
WEEK4_PROCESSED_DIR = PROJECT_ROOT / "Data" / "processed" / "week4"

module_path = SRC_DIR / "week4_src_model_ready.py"
spec = importlib.util.spec_from_file_location("week4_model_ready", module_path)
week4 = importlib.util.module_from_spec(spec)
sys.modules["week4_model_ready"] = week4
spec.loader.exec_module(week4)

print("Project root:", PROJECT_ROOT)
print("Week 4 source:", module_path)
print("Week 4 reports:", WEEK4_REPORTS_DIR)

## 2. Rebuilds analytical table without rewriting prior-week reports

In [ ]:
monthly = week4.rebuild_monthly_without_rewriting_prior_reports(PROJECT_ROOT)
monthly = week4.add_past_only_features(monthly)

print(f"Carrier-route-month rows: {len(monthly):,}")
print(f"Reference period: {monthly['Period'].min():%Y-%m} through {monthly['Period'].max():%Y-%m}")
print("Past-only engineered columns present:")
print([c for c in ["PassengerLag1", "PassengerLag2", "PassengerRolling3Mean", "ScheduleCompletionRate"] if c in monthly.columns])

## 3. Recreate Week 3 chronological partitions

In [ ]:
(
    model_data,
    train_df,
    validation_df,
    test_df,
    split_summary,
) = week4.rebuild_week3_splits(monthly, PROJECT_ROOT)

display(split_summary)

print("Chronological integrity:")
print("Training < validation:", train_df[week4.TARGET_TIME_COLUMN].max() < validation_df[week4.TARGET_TIME_COLUMN].min())
print("Validation < test:", validation_df[week4.TARGET_TIME_COLUMN].max() < test_df[week4.TARGET_TIME_COLUMN].min())

## 4. Fit preprocessing on training data only. Apply unchanged to validation/test

Pipeline performs the following processing:

1. Treat invalid load factors and defensively flag invalid negative numeric values.
2. Fit numeric medians on training data only and impute missing model inputs.
3. Create exact-calendar passenger lags and a 3-month rolling mean with missing-history indicators.
4. Apply `log1p` to strongly skewed demand/capacity/count features.
5. Frequency-encode carrier, origin, and destination using training frequencies only.
6. Add cyclical forecast-month features and a deterministic time trend.
7. Standardize numeric model inputs using training-only mean/standard deviation.
8. Preserve both the raw target and `TargetLog1p` for later model comparison.

In [ ]:
results = week4.run_week4_from_splits(
    train_df,
    validation_df,
    test_df,
    split_summary,
    project_root=PROJECT_ROOT,
    save=True,
    figures_raw_train=train_df,
)

print("Training model-ready shape:", results.train_model_ready.shape)
print("Validation model-ready shape:", results.validation_model_ready.shape)
print("Test model-ready shape:", results.test_model_ready.shape)
print("Model feature count:", len(week4.model_feature_columns(results.train_model_ready)))

## 5. Processing summary

In [ ]:
display(results.processing_summary)

## 6. Model-ready feature manifest and sample

- Identifiers and date fields remain available for auditability. 
- They'r excluded from numeric model-feature list. 
- Model features themselves contain only numeric values.

In [ ]:
display(results.feature_manifest)

feature_cols = week4.model_feature_columns(results.train_model_ready)
print("Model features:")
print(feature_cols)

display(results.train_model_ready.head())

## 7. Missingness and invalid-value treatment audit

In [ ]:
display(results.missingness_before_after)

## 8. Holdout-category audit

- Frequency maps learned from training only. 
- Any category that first appears in validation or test receives frequency `0.0`. This stops causing error or being learned from future data.

In [ ]:
display(results.unknown_category_audit)

## 9. Training-fitted preprocessing parameters

- Documents medians / scaling coefficients learned from training partition. 
- Validation and test data are transformed using these frozen values.

In [ ]:
display(results.preprocessor_parameters)

## 10. Data-quality checks on all splits

In [ ]:
display(results.data_quality_checks)
assert results.data_quality_checks["Passed"].all(), "At least 1 Week4 quality check failed."

## 11. Week 4 figures

In [ ]:
figure_paths = [
    WEEK4_FIGURES_DIR / "week4_figure_training_target_log1p.png",
    WEEK4_FIGURES_DIR / "week4_figure_training_missing_before_imputation.png",
    WEEK4_FIGURES_DIR / "week4_figure_split_row_retention.png",
    WEEK4_FIGURES_DIR / "week4_figure_holdout_unknown_categories.png",
]

for path in figure_paths:
    if path.exists():
        print(path.name)
        display(Image(filename=str(path)))

## 12. Week 4 artifacts

Only representative processed-data samples pushed to GitHub.

In [ ]:
print("Week 4 tables:")
for path in sorted(WEEK4_TABLES_DIR.glob("*")):
    print("-", path.name)

print("\nWeek 4 figures:")
for path in sorted(WEEK4_FIGURES_DIR.glob("*.png")):
    print("-", path.name)

print("\nProcessed-data samples:")
for path in sorted(WEEK4_PROCESSED_DIR.glob("*")):
    print("-", path.name)

## 13. Week 4 completion checks

In [ ]:
assert len(results.train_model_ready) == int(split_summary.loc[split_summary["Split"].eq("Training"), "Rows"].iloc[0])
assert len(results.validation_model_ready) == int(split_summary.loc[split_summary["Split"].eq("Validation"), "Rows"].iloc[0])
assert len(results.test_model_ready) == int(split_summary.loc[split_summary["Split"].eq("Test"), "Rows"].iloc[0])

for frame in [results.train_model_ready, results.validation_model_ready, results.test_model_ready]:
    features = week4.model_feature_columns(frame)
    assert not frame[features].isna().any().any()

assert (WEEK4_TABLES_DIR / "week4_preprocessing_metadata.json").exists()
assert (WEEK4_FIGURES_DIR / "week4_figure_training_target_log1p.png").exists()
assert (WEEK4_PROCESSED_DIR / "week4_training_model_ready_sample.csv").exists()

print("All Week 4 checks passed.")
print("===== WEEK 4 END =====")